# Week 5 Day 1 — Bond Residuals (Richness/Cheapness)

For each maturity grid point and each date, compute:

```
residual_bps = (observed_rate − fitted_rate) × 10,000
```

- **observed_rate** — Fed's published GSW zero yield (`sveny01`–`sveny30`)
- **fitted_rate** — our own Svensson model evaluated at that maturity
- **Positive** → cheap (bond yields more than the curve predicts)
- **Negative** → rich (bond yields less than the curve predicts)

Two cleaning steps applied inside `compute_residuals()` before saving:

1. **Outlier filter** — rows where `|residual_bps| > 30` are dropped entirely.
   These are Svensson parameter-instability days, not real trading opportunities.
2. **Per-maturity demeaning** — the full-history mean residual at each maturity is
   subtracted so the signal fluctuates around zero. The 30Y had a −8 bp structural
   bias from the Svensson long-end misfit; demeaning removes it. The constants are
   stored in `residual_means.parquet` so Day 3 can apply the same shift consistently.

Outputs:
- `data/processed/bond_residuals.parquet` — long format, one row per (date, maturity), demeaned
- `data/processed/residual_means.parquet` — per-maturity demeaning constants

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.signals.richness import compute_residuals

## 1. Compute and save residuals

In [ ]:
residuals = compute_residuals()
print(f'Shape: {residuals.shape}')
print(f'Date range: {residuals["date"].min().date()} → {residuals["date"].max().date()}')
print(f'Maturities: {sorted(residuals["maturity"].unique())}')
residuals.head(8)

## 2. Demeaning constants

These are the per-maturity mean residuals subtracted from every row. Day 3 loads
this file and applies the same shift before z-scoring so there is no look-ahead.

In [ ]:
means = pd.read_parquet('../data/processed/residual_means.parquet')
means = means.set_index('maturity')
means.round(2)

## 3. Summary statistics by maturity

After outlier filtering and demeaning, all means should be ≈ 0 by construction.
The `std` column is the quantity that matters: it tells you how much a maturity
point typically deviates from its structural level. The long end (20Y, 30Y) is
noisier — that feeds into wider z-score thresholds in Day 3.

In [ ]:
stats = (
    residuals
    .groupby('maturity')['residual_bps']
    .agg(['mean', 'std', 'min', 'max',
          lambda s: (s > 0).mean()])
    .rename(columns={'<lambda_0>': 'pct_cheap'})
    .round(2)
)
stats

## 4. Residuals through time

Plot the demeaned residual at each maturity over the full history. A few things to look for:

- **Clustering near zero** → demeaning worked; no persistent structural bias remains
- **Large spikes** → any remaining instability days that fell within the 30 bp threshold
- **Mean-reverting noise** → exactly what a relative-value signal needs

In [ ]:
maturities = sorted(residuals['maturity'].unique())
pivot = residuals.pivot(index='date', columns='maturity', values='residual_bps')

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for i, mat in enumerate(maturities):
    ax = axes[i]
    ax.plot(pivot.index, pivot[mat], lw=0.6, color='steelblue')
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(f'{mat}Y  (std = {pivot[mat].std():.1f} bp)')
    ax.set_ylabel('residual (bp)')

fig.suptitle('Observed − Fitted zero rate (bp)  |  positive = cheap', y=1.01)
plt.tight_layout()
plt.savefig('data/week5_day1_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Single-date snapshot

For the most recent date, plot the observed curve vs. the fitted curve and mark
the residuals. This is the visual that will appear in the tearsheet.

In [ ]:
latest_date = residuals['date'].max()
snap = residuals[residuals['date'] == latest_date].sort_values('maturity')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(snap['maturity'], snap['observed_rate'] * 100, 'o-', label='Observed (Fed GSW)', color='steelblue')
ax1.plot(snap['maturity'], snap['fitted_rate']   * 100, 's--', label='Fitted (our Svensson)', color='firebrick')
ax1.set_ylabel('Zero rate (%)')
ax1.set_title(f'Yield curve snapshot — {latest_date.date()}')
ax1.legend()

colors = ['firebrick' if r < 0 else 'steelblue' for r in snap['residual_bps']]
ax2.bar(snap['maturity'], snap['residual_bps'], color=colors, width=0.6)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Residual (bp)')
ax2.set_xlabel('Maturity (years)')
ax2.set_title('Residual = Observed − Fitted  |  blue = cheap, red = rich')
ax2.set_xticks(snap['maturity'])

plt.tight_layout()
plt.savefig('data/week5_day1_snapshot.png', dpi=150, bbox_inches='tight')
plt.show()